In [2]:
import pandas as pd


In [9]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(
    "../data/raw/complaint_embeddings.parquet"
)

print("Rows:", parquet_file.metadata.num_rows)
print("Row groups:", parquet_file.num_row_groups)
print("Columns:", parquet_file.schema.names)

Rows: 1375327
Row groups: 2
Columns: ['id', 'document', 'element', 'chunk_index', 'company', 'complaint_id', 'date_received', 'issue', 'product', 'product_category', 'state', 'sub_issue', 'total_chunks']


In [13]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile("../data/raw/complaint_embeddings.parquet")

for batch in parquet_file.iter_batches(
    batch_size=5,
    columns=["document"]
):
    df = batch.to_pandas()
    print(df.head())
    break

                                            document
0  a card was opened under my name by a fraudster...
1  i made the mistake of using my wellsfargo debi...
2  and got a letter stating my dispute was reject...
3  dear cfpb, i have a secured credit card with c...
4  y confirmation whatsoever to report to the pol...


In [14]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [16]:
index = faiss.read_index("../vector_store/complaints_faiss.index")

metadata_df = pd.read_parquet("../vector_store/chunks_metadata.parquet")

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 623.60it/s]


In [91]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 786.00it/s]


In [93]:
def retrieve_chunks(query, k=5):
    """
    Takes a user question and returns top-k relevant complaint chunks
    """

    # Step 1: Convert query to embedding
    query_embedding = embedding_model.encode([query])

    query_embedding = np.array(
        query_embedding,
        dtype=np.float32
    )

    # Step 2: Search FAISS
    distances, indices = index.search(query_embedding, k)

    # Step 3: Get results from metadata
    results = metadata_df.iloc[indices[0]].copy()

    # Step 4: Attach similarity scores
    results["distance"] = distances[0]

    return results

In [94]:
results = retrieve_chunks("unauthorized credit card charges")

for _, row in results.iterrows():
    print("\n" + "="*80)
    print("Product:", row["product"])
    print("Issue:", row["issue"])
    print("Company:", row["company"])
    print("Distance:", row["distance"])



Product: Credit card
Issue: Fees or interest
Company: JPMORGAN CHASE & CO.
Distance: 0.440578818321228

Product: Credit card
Issue: Problem with a purchase shown on your statement
Company: BANK OF AMERICA, NATIONAL ASSOCIATION
Distance: 0.5077540874481201

Product: Money transfer, virtual currency, or money service
Issue: Unauthorized transactions or other transaction problem
Company: Block, Inc.
Distance: 0.5201938152313232

Product: Credit card
Issue: Problem with a purchase shown on your statement
Company: KEYCORP
Distance: 0.5246365070343018

Product: Credit card
Issue: Other features, terms, or problems
Company: ENOVA INTERNATIONAL, INC.
Distance: 0.5461913347244263


A retrieval function was implemented that converts user queries into embeddings using SentenceTransformer, searches the FAISS index for nearest neighbors, and returns the top-k most relevant complaint chunks along with their metadata.

In [115]:
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="distilgpt2"
)

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 1019.45it/s]


In [109]:
prompt = f"""
Answer the question using the context.

You must:
- Be factual
- Use only the context
- Write ONE sentence only
- Do NOT repeat phrases

Context:
{context}

Question:
{query}

Answer:
"""

In [112]:
def generate_answer(query, k=5):

    # Step 1: Retrieve relevant chunks
    results = retrieve_chunks(query, k)

    # Step 2: Build context
    context = "\n\n".join(results["text"].tolist()[:3])

    # Step 3: Create prompt (simple + stable for GPT2)
    prompt = f"""
You are a financial complaint analyst.

Use ONLY the context below to answer the question in ONE clear sentence.

Context:
{context}

Question:
{query}

Answer:
"""

    # Step 4: Generate response
    response = llm(
    prompt,
    max_new_tokens=30,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.3,
    return_full_text=False
)[0]["generated_text"]

    # Step 5: Clean output
    response = response.strip()

    return response

In [113]:
query = "Why are customers complaining about credit cards?"

print(generate_answer(query))

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'do_sample', 'temperature', 'top_p', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


They make sure we give our consumer protection information on how much money I owe, all my bills taken out as cash has been given back because after every


In [96]:
questions = [
    "Why are customers complaining about credit cards?",
    "What issues do customers face with banking accounts?",
    "Why do customers report unauthorized credit card activity?",
    "What problems occur with customer service in financial institutions?",
    "Why are customers unhappy with billing statements?",
    "What complaints are common in money transfer services?",
    "Why do customers complain about account closures?",
    "Why do customers complain about fees and interest charges?"
]

In [57]:
!pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [116]:
import pandas as pd

results_list = []

for q in questions:
    retrieved = retrieve_chunks(q)

    answer = generate_answer(q)

    sources = retrieved["text"].tolist()[:2]

    results_list.append({
        "Question": q,
        "Generated Answer": answer,
        "Retrieved Sources": sources,
        "Quality Score (1-5)": None,   
        "Comments/Analysis": None      
    })

df_eval = pd.DataFrame(results_list)
df_eval
print(df_eval.to_markdown(index=False))
print(generate_answer(q))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/t

| Question                                                             | Generated Answer                                                                                                                                                                     | Retrieved Sources                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [103]:
answer = generate_answer(
    "Why are customers complaining about credit cards?"
)

print("ANSWER:")
print(repr(answer))

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:
' (9-10 sentences,'


In [102]:
test = llm("What is the capital of France?")
print(test)

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'What is the capital of France?\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n'}]


In [76]:
print(llm.model.name_or_path)

google/flan-t5-base


In [79]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1982.82it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [81]:
prompt = """
Question: What is the capital of France?
Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

london


In [83]:
print(model.config._name_or_path)

prompt = "Question: What is the capital of France? Answer:"

inputs = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=20
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

google/flan-t5-base
london
